# 从完整 decoder 到可检查的训练

这个 Notebook 真正在 CPU 训练小模型，需要 PyTorch 2.8.0 CPU，安装命令见 [README](README.md)。语料是构造句子，loss 下降不代表泛化能力。

先看 [Attention 的矩阵算例](../../10-Knowledge/02-foundation-models/04-labs/01-tokenization-and-attention.ipynb)。这里不要求记住全部源码：先追踪一组 ID，再手动展开一次 Attention，核对答案标签，然后观察参数更新。每一步都能与 [model.py](src/tiny_transformer/model.py) 对上。


In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/run_python.py").exists())
sys.path.insert(0, str(ROOT / "20-Projects/tiny-transformer/src"))


In [2]:
import torch
from tiny_transformer.model import CharacterTokenizer, Config, Decoder, supervised_batch
torch.manual_seed(7);torch.set_num_threads(2)
tokenizer=CharacterTokenizer('hello answer')
model=Decoder(Config(len(tokenizer.vocabulary)))
x,y=supervised_batch(tokenizer,[('hello',' answer')])
logits,cache=model(x)
print({'input_ids':x.tolist(),'shifted_labels':y.tolist(),'logits_shape':list(logits.shape),'first_layer_key_shape':list(cache[0][0].shape)})

{'input_ids': [[1, 7, 6, 8, 8, 10, 4, 5, 9, 12, 13, 6, 11]], 'shifted_labels': [[-100, -100, -100, -100, -100, 4, 5, 9, 12, 13, 6, 11, 2]], 'logits_shape': [1, 13, 14], 'first_layer_key_shape': [1, 4, 13, 8]}


## 把第一层 Attention 展开一次

此时模型尚未训练，分数没有语言含义。我们只检查运算本身：4 个头、每头 8 维；行是当前 query，列是可读取的 key。先看第 0 个头的第二行，未来位置权重应为 0，整行权重和应为 1。

`joined` 是各头的 Value 加权结果；经过输出投影后，才等于模块输出。这个比较能发现转置、head 拼接或 softmax 轴写错。


In [3]:
import math
with torch.no_grad():
    embeddings = model.tokens(x) + model.positions(torch.arange(x.shape[1]))
    block = model.blocks[0]
    hidden = block.norm1(embeddings)
    attn = block.attention
    q, k, v = attn.qkv(hidden).chunk(3, dim=-1)
    B, T, D = hidden.shape
    def split_heads(t):
        return t.reshape(B, T, attn.heads, attn.depth).transpose(1, 2)
    q, k, v = map(split_heads, (q, k, v))
    scores = q @ k.transpose(-2, -1) / math.sqrt(attn.depth)
    allowed = torch.tril(torch.ones(T, T, dtype=torch.bool))
    weights = scores.masked_fill(~allowed, float('-inf')).softmax(-1)
    joined = (weights @ v).transpose(1, 2).contiguous().reshape(B, T, D)
    manual = attn.projection(joined)
    actual, _ = attn(hidden)
print('Q/K/V shape:', list(q.shape), 'scores shape:', list(scores.shape))
print('head 0, query 1 weights:', weights[0, 0, 1].tolist())
print('manual vs module max error:', (manual - actual).abs().max().item())
assert torch.allclose(weights.sum(-1), torch.ones(B, attn.heads, T))
assert torch.equal(weights[..., ~allowed], torch.zeros_like(weights[..., ~allowed]))
assert torch.allclose(manual, actual, atol=1e-6)


Q/K/V shape: [1, 4, 13, 8] scores shape: [1, 4, 13, 13]
head 0, query 1 weights: [0.46852484345436096, 0.5314751267433167, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
manual vs module max error: 0.0


## 答案第一个字符在哪个位置计损失

使用一份单独的小词表，prompt=`Q`、answer=`xy`。输入是 `BOS Q x y`，目标是 `Q x y EOS`；只屏蔽预测 Q 的位置，保留 Q 位置对 x 的预测。这里的 `-100` 是 PyTorch 的忽略标签值，不是词表中的 token。

先自行写出四格标签，再运行检查。完整模型采用右 padding，padding 的目标同样不计 loss；不能把这项约定直接套到左 padding 或打包样本。


In [4]:
label_tokenizer = CharacterTokenizer('Qxy')
label_x, label_y = supervised_batch(label_tokenizer, [('Q', 'xy')])
show = lambda row: ['忽略' if i == -100 else label_tokenizer.vocabulary[i] for i in row]
print('inputs:', show(label_x[0].tolist()))
print('labels:', show(label_y[0].tolist()))
assert show(label_y[0].tolist()) == ['忽略', 'x', 'y', '<eos>']


inputs: ['<bos>', 'Q', 'x', 'y']
labels: ['忽略', 'x', 'y', '<eos>']


## 训练、LoRA、缓存和重载

所有结果来自本次运行。检查冻结权重没有更新，再看 loss；只看 loss 会漏掉误解冻问题。

In [5]:
from tiny_transformer.experiment import experiment
with tempfile.TemporaryDirectory() as directory:
    report=experiment(directory,steps=65)
assert report['pretraining_loss'][1]<report['pretraining_loss'][0]
assert report['frozen_weights_unchanged']
print(json.dumps(report,ensure_ascii=False,indent=2))

{
  "seed": 7,
  "device": "cpu",
  "torch": "2.8.0+cpu",
  "parameters": 31464,
  "pretraining_loss": [
    3.253199338912964,
    0.0016245456645265222
  ],
  "sft_loss": [
    3.8780176639556885,
    0.003843255341053009
  ],
  "sft_trainable_parameters": [
    {
      "name": "head.a",
      "shape": [
        4,
        32
      ],
      "numel": 128
    },
    {
      "name": "head.b",
      "shape": [
        26,
        4
      ],
      "numel": 104
    }
  ],
  "sft_prediction_ids_before": [
    [
      22,
      18,
      18,
      23,
      15,
      14,
      5,
      4,
      22,
      15,
      5,
      24,
      24,
      15,
      6,
      6,
      5,
      22
    ],
    [
      22,
      18,
      18,
      23,
      18,
      15,
      11,
      19,
      23,
      5,
      23,
      24,
      5,
      18,
      6,
      21,
      5,
      22
    ]
  ],
  "sft_prediction_ids_after": [
    [
      2,
      16,
      5,
      5,
      6,
      6,
      4,
      4,
     

## 先判断报告对应哪个阶段

| 观察 | 它能说明什么 | 下一步检查 |
|---|---|---|
| `pretraining_loss` 下降 | 构造语料可被拟合 | 另划未见过的数据才能讨论泛化 |
| `frozen_weights_unchanged=True` | SFT 没误更新基座 | `sft_trainable_parameters` 只含 head.a/head.b |
| `sample` | 预训练后的一段贪心输出 | 它在 SFT/DPO 之前生成 |
| `sft_prediction_ids_after` | 有正确前缀时各位置的预测 | 不等于自由生成时会一直给自己正确前缀 |
| `dpo_loss` 下降 | 当前模型与冻结 reference 的偏好差值改善 | 只用一组构造偏好，未评测通用偏好能力 |
| `policy_probabilities_after` | 独立两动作策略做过一次更新 | 它不是字符语言模型的 PPO 训练结果 |

`steps=65` 与仓库存档的默认 100 步不同，loss 数字不应完全一致。上面使用临时目录，离开 `with` 后文件被删除；想保留自己训练的权重，运行 README 的 `--output` 命令。


## 修改一个条件再观察

在源码中查看 Attention.forward 的 prefix 和 mask。缓存时位置必须加上旧前缀长度。先理解为什么，再在副本中故意删除偏移，运行 tests/test_decoder.py，观察缓存一致性测试失败。不要把故意破坏的副本当作正确实现提交。